IMPORT CÁC THƯ VIỆN VÀO

In [1]:
import pandas as pd
import numpy as numpy

1. ĐỌC DỮ LIỆU ĐẦU VÀo

In [2]:
file_path = r"D:\TT Dữ liệu trực quan\ProjectCuoiKy\Dataset\AQI — EPA AirData\AQI_Respiratory_Disease_2019_2022_Merged_Clean.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (7566, 24)


,CountyFIPS,County,StateAbbr,State,Year,Population,Geolocation,Days_with_AQI,Good_Days,Moderate_Days,...,Max_AQI,AQI_90th_Percentile,Median_AQI,Days_NO2,Days_Ozone,Days_PM2_5,Days_PM10,Smoking_AdjPrev,Disease,Disease_AdjPrev
0,1003,Baldwin,AL,Alabama,2019,"223,234",POINT (-87.72275422 30.72811673),271,221,50,...,80,55,40,0,198,73,0,19.9,COPD,6.9
1,1003,Baldwin,AL,Alabama,2019,"223,234",POINT (-87.72275422 30.72811673),271,221,50,...,80,55,40,0,198,73,0,19.9,Current Asthma,8.9
2,1027,Clay,AL,Alabama,2019,"13,235",POINT (-85.86076142 33.26916978),107,75,32,...,71,56,41,0,0,107,0,25.6,COPD,10.1
3,1027,Clay,AL,Alabama,2019,"13,235",POINT (-85.86076142 33.26916978),107,75,32,...,71,56,41,0,0,107,0,25.6,Current Asthma,10.6
4,1033,Colbert,AL,Alabama,2019,"55,241",POINT (-87.80480288 34.70084917),263,238,25,...,65,50,38,0,219,44,0,21.5,COPD,8.2


2. KIỂM TRA CẤU TRÚC DỮ LIỆU

In [3]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Columns:
['CountyFIPS', 'County', 'StateAbbr', 'State', 'Year', 'Population', 'Geolocation', 'Days_with_AQI', 'Good_Days', 'Moderate_Days', 'USG_Days', 'Unhealthy_Days', 'Very_Unhealthy_Days', 'Hazardous_Days', 'Max_AQI', 'AQI_90th_Percentile', 'Median_AQI', 'Days_NO2', 'Days_Ozone', 'Days_PM2_5', 'Days_PM10', 'Smoking_AdjPrev', 'Disease', 'Disease_AdjPrev']

Data types:
CountyFIPS               int64
County                     str
StateAbbr                  str
State                      str
Year                     int64
Population                 str
Geolocation                str
Days_with_AQI            int64
Good_Days                int64
Moderate_Days            int64
USG_Days                 int64
Unhealthy_Days           int64
Very_Unhealthy_Days      int64
Hazardous_Days           int64
Max_AQI                  int64
AQI_90th_Percentile      int64
Median_AQI               int64
Days_NO2                 int64
Days_Ozone               int64
Days_PM2_5               int64
Days_P

3. KIỂM TRA MISSING VALUE VÀ GIÁ TRỊ TRÙNG LẶP

In [4]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print(
    "Duplicate County-Year-Disease:",
    df.duplicated(subset=["CountyFIPS", "Year", "Disease"]).sum()
)

Missing values:
CountyFIPS             0
County                 0
StateAbbr              0
State                  0
Year                   0
Population             0
Geolocation            0
Days_with_AQI          0
Good_Days              0
Moderate_Days          0
USG_Days               0
Unhealthy_Days         0
Very_Unhealthy_Days    0
Hazardous_Days         0
Max_AQI                0
AQI_90th_Percentile    0
Median_AQI             0
Days_NO2               0
Days_Ozone             0
Days_PM2_5             0
Days_PM10              0
Smoking_AdjPrev        0
Disease                0
Disease_AdjPrev        0
dtype: int64

Duplicate rows: 0
Duplicate County-Year-Disease: 0


4. CHUẨN HÓA DỮ LIỆU

In [5]:
#4.1 Chuẩn hóa biến population
df["Population"] = (
    df["Population"]
    .str.replace(",", "", regex=False)
    .astype("int64")
)

print("Population type:", df["Population"].dtype)
print(df["Population"].head())

Population type: int64
0    223234
1    223234
2     13235
3     13235
4     55241
Name: Population, dtype: int64


In [6]:
# 4.2 Kiểm tra giá trị không hợp lệ

day_cols = [
    "Days_with_AQI", "Good_Days", "Moderate_Days",
    "USG_Days", "Unhealthy_Days",
    "Very_Unhealthy_Days", "Hazardous_Days",
    "Days_NO2", "Days_Ozone", "Days_PM2_5", "Days_PM10"
]

print("Population <= 0:", (df["Population"] <= 0).sum())
print("Số ngày ngoài khoảng 0-366:", ((df[day_cols] < 0) | (df[day_cols] > 366)).sum().sum())
print("Smoking ngoài 0-100:", ((df["Smoking_AdjPrev"] < 0) | (df["Smoking_AdjPrev"] > 100)).sum())
print("Disease ngoài 0-100:", ((df["Disease_AdjPrev"] < 0) | (df["Disease_AdjPrev"] > 100)).sum())

Population <= 0: 0
Số ngày ngoài khoảng 0-366: 0
Smoking ngoài 0-100: 0
Disease ngoài 0-100: 0


In [7]:
# 4.3 Kiểm tra tính nhất quán số ngày AQI

aqi_day_sum = (
    df["Good_Days"]
    + df["Moderate_Days"]
    + df["USG_Days"]
    + df["Unhealthy_Days"]
    + df["Very_Unhealthy_Days"]
    + df["Hazardous_Days"]
)

print("Không khớp Days_with_AQI:", (aqi_day_sum != df["Days_with_AQI"]).sum())

Không khớp Days_with_AQI: 0


5. KIỂM TRA OUTLIER BẰNG IQR

In [9]:
# 5.1 Kiểm tra outlier bằng IQR

outlier_cols = [
    "Population",
    "Days_with_AQI",
    "Max_AQI",
    "AQI_90th_Percentile",
    "Median_AQI",
    "Days_NO2",
    "Days_Ozone",
    "Days_PM2_5",
    "Days_PM10",
    "Smoking_AdjPrev",
    "Disease_AdjPrev"
]

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    print(f"\n--- {col} ---")
    print("Q1:", round(Q1, 2))
    print("Q3:", round(Q3, 2))
    print("IQR:", round(IQR, 2))
    print("Lower Bound:", round(lower_bound, 2))
    print("Upper Bound:", round(upper_bound, 2))
    print("Số Outlier:", len(outliers))


--- Population ---
Q1: 33608.5
Q3: 265860.0
IQR: 232251.5
Lower Bound: -314768.75
Upper Bound: 614237.25
Số Outlier: 816

--- Days_with_AQI ---
Q1: 336.0
Q3: 365.0
IQR: 29.0
Lower Bound: 292.5
Upper Bound: 408.5
Số Outlier: 1618

--- Max_AQI ---
Q1: 84.0
Q3: 133.0
IQR: 49.0
Lower Bound: 10.5
Upper Bound: 206.5
Số Outlier: 436

--- AQI_90th_Percentile ---
Q1: 50.0
Q3: 64.0
IQR: 14.0
Lower Bound: 29.0
Upper Bound: 85.0
Số Outlier: 460

--- Median_AQI ---
Q1: 35.0
Q3: 45.0
IQR: 10.0
Lower Bound: 20.0
Upper Bound: 60.0
Số Outlier: 526

--- Days_NO2 ---
Q1: 0.0
Q3: 0.0
IQR: 0.0
Lower Bound: 0.0
Upper Bound: 0.0
Số Outlier: 1202

--- Days_Ozone ---
Q1: 60.0
Q3: 244.0
IQR: 184.0
Lower Bound: -216.0
Upper Bound: 520.0
Số Outlier: 0

--- Days_PM2_5 ---
Q1: 8.0
Q3: 251.0
IQR: 243.0
Lower Bound: -356.5
Upper Bound: 615.5
Số Outlier: 0

--- Days_PM10 ---
Q1: 0.0
Q3: 0.0
IQR: 0.0
Lower Bound: 0.0
Upper Bound: 0.0
Số Outlier: 1590

--- Smoking_AdjPrev ---
Q1: 14.2
Q3: 19.7
IQR: 5.5
Lower Bound: 5.9

Xử lý Outlier: Sau khi kiểm tra các biến định lượng bằng phương pháp IQR, chỉ Median_AQI, AQI_90th_Percentile, Smoking_AdjPrev và Disease_AdjPrev được lựa chọn để xử lý outlier vì đây là các biến chính phục vụ phân tích và mô hình dự đoán. Các biến như Population, Days_with_AQI, Max_AQI và số ngày theo từng chất ô nhiễm được giữ lại do các giá trị cực trị có thể phản ánh sự khác biệt thực tế giữa các khu vực. Riêng Days_with_AQI được đánh giá bằng AQI Coverage

In [10]:
# 5.2 Xử lý outlier

df_original = df.copy()

cols_remove = [
    "Median_AQI",
    "AQI_90th_Percentile",
    "Smoking_AdjPrev",
    "Disease_AdjPrev"
]

outlier_mask = pd.Series(False, index=df.index)

for col in cols_remove:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask |= (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    )

df = df[~outlier_mask].copy()

print("Trước xử lý:", len(df_original))
print("Outlier bị loại:", outlier_mask.sum())
print("Sau xử lý:", len(df))
print("Tỷ lệ loại:", round(outlier_mask.mean() * 100, 2), "%")

Trước xử lý: 7566
Outlier bị loại: 776
Sau xử lý: 6790
Tỷ lệ loại: 10.26 %


6. KIỂM TRA ĐỘ PHỦ DỮ LIỆU AQI

In [11]:
# 6.1 Tính độ phủ dữ liệu AQI

df["Days_in_Year"] = df["Year"].apply(
    lambda year: 366 if year % 4 == 0 else 365
)

df["AQI_Coverage_Pct"] = (
    df["Days_with_AQI"] / df["Days_in_Year"] * 100
)

print(df["AQI_Coverage_Pct"].describe())

count    6790.000000
mean       90.311606
std        18.230696
min         1.917808
25%        93.702934
50%        99.180328
75%       100.000000
max       100.000000
Name: AQI_Coverage_Pct, dtype: float64


In [12]:
# 6.2 Kiểm tra mức độ phủ AQI

print("Coverage < 50%:", (df["AQI_Coverage_Pct"] < 50).sum())
print("Coverage < 75%:", (df["AQI_Coverage_Pct"] < 75).sum())
print("Coverage < 90%:", (df["AQI_Coverage_Pct"] < 90).sum())
print("Coverage >= 90%:", (df["AQI_Coverage_Pct"] >= 90).sum())

Coverage < 50%: 352
Coverage < 75%: 1212
Coverage < 90%: 1532
Coverage >= 90%: 5258


In [13]:
# 6.3 Lọc dữ liệu có độ phủ AQI >= 50%

rows_before = len(df)

df = df[df["AQI_Coverage_Pct"] >= 50].copy()

rows_after = len(df)

print("Trước khi lọc:", rows_before)
print("Số dòng bị loại:", rows_before - rows_after)
print("Sau khi lọc:", rows_after)
print("Tỷ lệ loại:", round((rows_before - rows_after) / rows_before * 100, 2), "%")

Trước khi lọc: 6790
Số dòng bị loại: 352
Sau khi lọc: 6438
Tỷ lệ loại: 5.18 %


7. KIỂM TRA TÍNH ĐẦY ĐỦ THEO NĂM

In [14]:
# 7.1 Kiểm tra số năm dữ liệu của mỗi County

county_year_count = df.groupby("CountyFIPS")["Year"].nunique()

print("Tổng số County:", len(county_year_count))

print("\nSố County theo số năm dữ liệu:")
print(
    county_year_count
    .value_counts()
    .sort_index(ascending=False)
)

Tổng số County: 898

Số County theo số năm dữ liệu:
Year
4    635
3    187
2     42
1     34
Name: count, dtype: int64


In [15]:
# 7.2 Đánh dấu County có đủ dữ liệu 4 năm

county_year_count = df.groupby("CountyFIPS")["Year"].nunique()

complete_counties = county_year_count[
    county_year_count == 4
].index

df["Complete_4_Years"] = df["CountyFIPS"].isin(complete_counties)

print(df["Complete_4_Years"].value_counts())
print("\nShape:", df.shape)

Complete_4_Years
True     5080
False    1358
Name: count, dtype: int64

Shape: (6438, 27)


8. BỔ SUNG BIẾN CÁC THÀNH PHỐ LỚN Ở MỸ

In [16]:
# 8.1 Đọc dữ liệu AQS Sites

aqs_sites = pd.read_csv("aqs_sites.csv", low_memory=False)

print("Shape:", aqs_sites.shape)
print("\nColumns:")
print(aqs_sites.columns.tolist())

Shape: (20994, 28)

Columns:
['State Code', 'County Code', 'Site Number', 'Latitude', 'Longitude', 'Datum', 'Elevation', 'Land Use', 'Location Setting', 'Site Established Date', 'Site Closed Date', 'Met Site State Code', 'Met Site County Code', 'Met Site Site Number', 'Met Site Type', 'Met Site Distance', 'Met Site Direction', 'GMT Offset', 'Owning Agency', 'Local Site Name', 'Address', 'Zip Code', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Tribe Name', 'Extraction Date']


Giải thích biến CountyFIPS

CountyFIPS là mã định danh duy nhất cho mỗi county (quận) tại Hoa Kỳ theo hệ thống mã FIPS. Mã này được hình thành từ:

State Code: mã của bang.
County Code: mã của county trong bang đó.

Trong bộ dữ liệu chính đã có biến CountyFIPS, trong khi file aqs_sites.csv lưu riêng State Code và County Code. Vì vậy, cần tạo CountyFIPS cho dữ liệu AQS Sites để có một khóa chung (key) giữa hai bộ dữ liệu.

Mục đích chính là:

Dataset chính (CountyFIPS) -> AQS Sites (CountyFIPS) -> xác định City Name tương ứng

Nhờ đó có thể bổ sung thông tin City vào bộ dữ liệu chính mà không phải ghép theo tên County hoặc State, giúp việc liên kết dữ liệu chính xác và nhất quán hơn.

In [17]:
# 8.2 Tạo CountyFIPS và kiểm tra mức độ khớp

aqs_sites["CountyFIPS"] = (
    pd.to_numeric(aqs_sites["State Code"], errors="coerce") * 1000
    + pd.to_numeric(aqs_sites["County Code"], errors="coerce")
)

aqs_sites["CountyFIPS"] = pd.to_numeric(
    aqs_sites["CountyFIPS"], errors="coerce"
).astype("Int64")

df["CountyFIPS"] = pd.to_numeric(
    df["CountyFIPS"], errors="coerce"
).astype("Int64")

data_counties = set(df["CountyFIPS"].dropna())
aqs_counties = set(aqs_sites["CountyFIPS"].dropna())

matched_counties = data_counties.intersection(aqs_counties)

print("County trong dataset chính:", len(data_counties))
print("County trong AQS Sites:", len(aqs_counties))
print("County khớp:", len(matched_counties))
print("Tỷ lệ khớp:",
      round(len(matched_counties) / len(data_counties) * 100, 2), "%")

print("\nSố dòng dataset chính:", len(df))
print("Số dòng có CountyFIPS khớp AQS:",
      df["CountyFIPS"].isin(aqs_counties).sum())
print("Số dòng không khớp:",
      (~df["CountyFIPS"].isin(aqs_counties)).sum())

County trong dataset chính: 898
County trong AQS Sites: 2133
County khớp: 898
Tỷ lệ khớp: 100.0 %

Số dòng dataset chính: 6438
Số dòng có CountyFIPS khớp AQS: 6438
Số dòng không khớp: 0


Làm sạch thông tin City: Sau khi liên kết bằng CountyFIPS, dữ liệu AQS Sites được giới hạn còn các county xuất hiện trong bộ dữ liệu chính. Biến City Name được chuẩn hóa bằng cách loại khoảng trắng thừa và xác định các bản ghi có tên thành phố hợp lệ. Các giá trị như 'Not in a city', 'Not in city' hoặc giá trị thiếu không được xem là thông tin thành phố hợp lệ.

In [18]:
# 8.3 Làm sạch thông tin City

aqs_relevant = aqs_sites[
    aqs_sites["CountyFIPS"].isin(data_counties)
].copy()

aqs_relevant["City_Clean"] = (
    aqs_relevant["City Name"]
    .astype("string")
    .str.strip()
)

aqs_relevant["Has_City"] = (
    aqs_relevant["City_Clean"].notna()
    & ~aqs_relevant["City_Clean"].str.lower().isin([
        "not in a city",
        "not in city"
    ])
)

city_check = (
    aqs_relevant
    .groupby("CountyFIPS")["Has_City"]
    .any()
)

print("County có thông tin City:", city_check.sum())
print("County không có thông tin City:", (~city_check).sum())
print("Tổng:", len(city_check))

County có thông tin City: 818
County không có thông tin City: 80
Tổng: 898


In [19]:
# 8.4 Kiểm tra số City trong mỗi County

aqs_city = aqs_relevant[
    aqs_relevant["Has_City"] == True
].copy()

county_city_check = (
    aqs_city
    .groupby("CountyFIPS")["City_Clean"]
    .nunique()
)

print("County có 1 City:",
      (county_city_check == 1).sum())

print("County có nhiều hơn 1 City:",
      (county_city_check > 1).sum())

print("Số City khác nhau trong AQS:",
      aqs_city["City_Clean"].nunique())

County có 1 City: 279
County có nhiều hơn 1 City: 539
Số City khác nhau trong AQS: 2611


Lựa chọn City đại diện: Do một County có thể chứa nhiều City, mỗi County được gán một City đại diện dựa trên số lượng trạm quan trắc AQS. City có số trạm AQS nhiều nhất trong County được chọn làm City đại diện. Cách này giúp đảm bảo mỗi CountyFIPS chỉ tương ứng với một City trước khi merge, tránh làm tăng số dòng của bộ dữ liệu chính.

In [20]:
# 8.5 Chọn City đại diện cho mỗi County

city_counts = (
    aqs_city
    .groupby(["CountyFIPS", "City_Clean"])
    .size()
    .reset_index(name="Num_Sites")
)

city_counts = city_counts.sort_values(
    ["CountyFIPS", "Num_Sites"],
    ascending=[True, False]
)

county_city = (
    city_counts
    .drop_duplicates(subset="CountyFIPS")
    [["CountyFIPS", "City_Clean", "Num_Sites"]]
    .rename(columns={
        "City_Clean": "City",
        "Num_Sites": "City_AQS_Sites"
    })
)

print("Số County có City đại diện:", len(county_city))
print("Số City đại diện khác nhau:", county_city["City"].nunique())

county_city.head(10)

Số County có City đại diện: 818
Số City đại diện khác nhau: 755


,CountyFIPS,City,City_AQS_Sites
0,1003,Fairhope,1
2,1033,Muscle Shoals,3
5,1049,Fort Payne,6
9,1051,Wetumpka,4
11,1055,Gadsden,7
13,1069,Dothan,4
15,1073,Birmingham,41
27,1089,Huntsville,19
32,1097,Mobile,30
36,1101,Montgomery,8


Bổ sung City vào dữ liệu chính: Sau khi xác định một City đại diện cho mỗi County, dữ liệu City được liên kết với bộ dữ liệu chính thông qua CountyFIPS. Sử dụng left join để giữ nguyên toàn bộ dữ liệu hiện có và kiểm tra các County chưa xác định được City đại diện.

In [21]:
# 8.6 Merge City vào dataset chính

rows_before = len(df)

df_city = df.merge(
    county_city[["CountyFIPS", "City"]],
    on="CountyFIPS",
    how="left"
)

print("Số dòng trước merge:", rows_before)
print("Số dòng sau merge:", len(df_city))
print("Số dòng có City:", df_city["City"].notna().sum())
print("Số dòng không có City:", df_city["City"].isna().sum())

Số dòng trước merge: 6438
Số dòng sau merge: 6438
Số dòng có City: 5868
Số dòng không có City: 570


Xử lý các quan sát không có City: Sau khi merge, có 570 quan sát không xác định được City đại diện từ AQS Sites. Vì biến City cần thiết cho mục tiêu trực quan hóa dữ liệu theo thành phố, các quan sát này được loại khỏi bộ dữ liệu cuối. Sau khi xử lý, dữ liệu vẫn đảm bảo số lượng quan sát tối thiểu theo yêu cầu.

In [22]:
# 8.7 Giữ các quan sát có City đại diện

df = df_city[df_city["City"].notna()].copy()

print("Shape:", df.shape)
print("Số County:", df["CountyFIPS"].nunique())
print("Số City:", df["City"].nunique())

print("\nSố dòng theo Disease:")
print(df["Disease"].value_counts())

print("\nSố dòng theo Year:")
print(df["Year"].value_counts().sort_index())

Shape: (5868, 28)
Số County: 818
Số City: 755

Số dòng theo Disease:
Disease
COPD              2934
Current Asthma    2934
Name: count, dtype: int64

Số dòng theo Year:
Year
2019    1516
2020    1510
2021    1438
2022    1404
Name: count, dtype: int64


9. TẠO BIẾN MỚI - BIẾN ĐÁNH GIÁ CHẤT LƯỢNG KHÔNG KHÍ AQI

Tạo biến Poor Air Days: Poor_Air_Days thể hiện tổng số ngày trong năm mà AQI thuộc các mức Unhealthy for Sensitive Groups (USG), Unhealthy, Very Unhealthy hoặc Hazardous. Biến Poor_Air_Days_Pct biểu diễn tỷ lệ các ngày chất lượng không khí kém trên tổng số ngày có dữ liệu AQI, giúp so sánh giữa các khu vực có số ngày quan trắc khác nhau.

In [23]:
# 9.1 Tạo biến Poor Air Days

df["Poor_Air_Days"] = (
    df["USG_Days"]
    + df["Unhealthy_Days"]
    + df["Very_Unhealthy_Days"]
    + df["Hazardous_Days"]
)

df["Poor_Air_Days_Pct"] = (
    df["Poor_Air_Days"] / df["Days_with_AQI"] * 100
)

print("Shape:", df.shape)
print("Missing Poor_Air_Days:", df["Poor_Air_Days"].isna().sum())
print("Missing Poor_Air_Days_Pct:", df["Poor_Air_Days_Pct"].isna().sum())

df[[
    "City",
    "Year",
    "Days_with_AQI",
    "Poor_Air_Days",
    "Poor_Air_Days_Pct"
]].head()

Shape: (5868, 30)
Missing Poor_Air_Days: 0
Missing Poor_Air_Days_Pct: 0


,City,Year,Days_with_AQI,Poor_Air_Days,Poor_Air_Days_Pct
0,Fairhope,2019,271,0,0.0
1,Fairhope,2019,271,0,0.0
2,Muscle Shoals,2019,263,0,0.0
3,Muscle Shoals,2019,263,0,0.0
4,Fort Payne,2019,361,0,0.0


Tách tọa độ địa lý: Biến Geolocation lưu tọa độ dưới dạng POINT (Longitude Latitude). Longitude và Latitude được tách thành hai biến số riêng để thuận tiện cho việc trực quan hóa dữ liệu trên bản đồ. Các tọa độ này đại diện cho vị trí địa lý của County trong dữ liệu gốc, không phải tọa độ trung tâm của City đại diện.

In [24]:
# 9.2 Tách Longitude và Latitude

coords = df["Geolocation"].str.extract(
    r"POINT \(([-\d.]+) ([-\d.]+)\)"
)

df["Longitude"] = pd.to_numeric(
    coords[0], errors="coerce"
)

df["Latitude"] = pd.to_numeric(
    coords[1], errors="coerce"
)

print("Missing Longitude:", df["Longitude"].isna().sum())
print("Missing Latitude:", df["Latitude"].isna().sum())
print("Shape:", df.shape)

df[[
    "City",
    "County",
    "State",
    "Latitude",
    "Longitude"
]].head()

Missing Longitude: 0
Missing Latitude: 0
Shape: (5868, 32)


,City,County,State,Latitude,Longitude
0,Fairhope,Baldwin,Alabama,30.728117,-87.722754
1,Fairhope,Baldwin,Alabama,30.728117,-87.722754
2,Muscle Shoals,Colbert,Alabama,34.700849,-87.804803
3,Muscle Shoals,Colbert,Alabama,34.700849,-87.804803
4,Fort Payne,DeKalb,Alabama,34.460203,-85.803597


10. CHỌN VÀ SẮP XẾP CÁC BIẾN CUỐI CÙNG 

Lựa chọn biến cuối cùng: Sau quá trình tiền xử lý, các biến cần thiết cho phân tích và trực quan hóa được giữ lại và sắp xếp theo từng nhóm thông tin. Các biến trung gian hoặc đã được chuyển đổi như Geolocation và Days_in_Year không được đưa vào bộ dữ liệu cuối.

In [26]:
# 10.1 Chọn các biến cho dataset cuối

columns_final = [
    "CountyFIPS",
    "City",
    "County",
    "StateAbbr",
    "State",
    "Latitude",
    "Longitude",
    "Year",
    "Population",

    "Days_with_AQI",
    "Good_Days",
    "Moderate_Days",
    "USG_Days",
    "Unhealthy_Days",
    "Very_Unhealthy_Days",
    "Hazardous_Days",

    "Max_AQI",
    "AQI_90th_Percentile",
    "Median_AQI",

    "Days_NO2",
    "Days_Ozone",
    "Days_PM2_5",
    "Days_PM10",

    "AQI_Coverage_Pct",
    "Poor_Air_Days",
    "Poor_Air_Days_Pct",

    "Smoking_AdjPrev",
    "Disease",
    "Disease_AdjPrev",

    "Complete_4_Years"
]

df_final = df[columns_final].copy()

print("Shape final:", df_final.shape)
print("\nColumns:")
print(df_final.columns.tolist())

Shape final: (5868, 30)

Columns:
['CountyFIPS', 'City', 'County', 'StateAbbr', 'State', 'Latitude', 'Longitude', 'Year', 'Population', 'Days_with_AQI', 'Good_Days', 'Moderate_Days', 'USG_Days', 'Unhealthy_Days', 'Very_Unhealthy_Days', 'Hazardous_Days', 'Max_AQI', 'AQI_90th_Percentile', 'Median_AQI', 'Days_NO2', 'Days_Ozone', 'Days_PM2_5', 'Days_PM10', 'AQI_Coverage_Pct', 'Poor_Air_Days', 'Poor_Air_Days_Pct', 'Smoking_AdjPrev', 'Disease', 'Disease_AdjPrev', 'Complete_4_Years']


11. KIỂM TRA LẠI DATASET CUỐI CÙNG

Kiểm tra dữ liệu cuối: Trước khi xuất dữ liệu, thực hiện kiểm tra lại giá trị thiếu, dòng trùng lặp, khóa County-Year-Disease, giá trị âm, tính nhất quán của số ngày AQI và phạm vi của các biến tỷ lệ.

In [27]:
# 11.1 Kiểm tra chất lượng dataset cuối

print("Shape:", df_final.shape)

print("\nTổng missing:",
      df_final.isnull().sum().sum())

print("Dòng trùng hoàn toàn:",
      df_final.duplicated().sum())

print("Trùng County-Year-Disease:",
      df_final.duplicated(
          subset=["CountyFIPS", "Year", "Disease"]
      ).sum())

numeric_check = [
    "Population",
    "Days_with_AQI",
    "Good_Days",
    "Moderate_Days",
    "USG_Days",
    "Unhealthy_Days",
    "Very_Unhealthy_Days",
    "Hazardous_Days",
    "Max_AQI",
    "AQI_90th_Percentile",
    "Median_AQI",
    "Days_NO2",
    "Days_Ozone",
    "Days_PM2_5",
    "Days_PM10",
    "Poor_Air_Days",
    "Poor_Air_Days_Pct"
]

print("\nSố giá trị âm:")
print((df_final[numeric_check] < 0).sum())

aqi_day_sum = (
    df_final["Good_Days"]
    + df_final["Moderate_Days"]
    + df_final["USG_Days"]
    + df_final["Unhealthy_Days"]
    + df_final["Very_Unhealthy_Days"]
    + df_final["Hazardous_Days"]
)

print("\nAQI category không khớp Days_with_AQI:",
      (aqi_day_sum != df_final["Days_with_AQI"]).sum())

print("Smoking ngoài 0-100:",
      ((df_final["Smoking_AdjPrev"] < 0) |
       (df_final["Smoking_AdjPrev"] > 100)).sum())

print("Disease ngoài 0-100:",
      ((df_final["Disease_AdjPrev"] < 0) |
       (df_final["Disease_AdjPrev"] > 100)).sum())

print("AQI Coverage ngoài 50-100:",
      ((df_final["AQI_Coverage_Pct"] < 50) |
       (df_final["AQI_Coverage_Pct"] > 100)).sum())

Shape: (5868, 30)

Tổng missing: 0
Dòng trùng hoàn toàn: 0
Trùng County-Year-Disease: 0

Số giá trị âm:
Population             0
Days_with_AQI          0
Good_Days              0
Moderate_Days          0
USG_Days               0
Unhealthy_Days         0
Very_Unhealthy_Days    0
Hazardous_Days         0
Max_AQI                0
AQI_90th_Percentile    0
Median_AQI             0
Days_NO2               0
Days_Ozone             0
Days_PM2_5             0
Days_PM10              0
Poor_Air_Days          0
Poor_Air_Days_Pct      0
dtype: int64

AQI category không khớp Days_with_AQI: 0
Smoking ngoài 0-100: 0
Disease ngoài 0-100: 0
AQI Coverage ngoài 50-100: 0


12. XUẤT DỮ LIỆU CUỐI CÙNG

In [28]:
# 12. Xuất dataset cuối

output_file = "AQI_Respiratory_Disease_Final.csv"

df_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu file:", output_file)
print("Shape:", df_final.shape)

Đã lưu file: AQI_Respiratory_Disease_Final.csv
Shape: (5868, 30)
